In [1]:
from core.scrapper import ScrapServidoresAtivos, TabularResource
import pandas as pd
import os
from typing import Generator

In [2]:
scraper = ScrapServidoresAtivos()

In [3]:
def filtrar_appggs(df:pd.DataFrame)->pd.DataFrame:

    if 'REF_CARGO_BAS' in df.columns:
        filtro = df['REF_CARGO_BAS'].str.lower().str.startswith('appgg')
    else:
        filtro = df['Cargo Base'].str.upper().str.startswith('ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENTAL')
    df_filtrado = df[filtro].reset_index(drop=True)

    return df_filtrado

In [4]:
def gerar_df_appggs(resource_gen:Generator[TabularResource], fname='dataset_appggs.csv')->pd.DataFrame:

    if os.path.exists(fname):
        return pd.read_csv(fname, sep=';')

    all_data = []
    for resource in resource_gen:
        df = resource.data
        df = filtrar_appggs(df)
        if df.shape[0] == 0:
            print(f'Nenhum APPP encontrado para o mes {resource.reference_date}')
            continue
        df['data_referencia'] = resource.reference_date
        df['ano_referencia'] = resource.reference_date.year
        df['mes_referencia'] = resource.reference_date.month

        all_data.append(df)

    df_final = pd.concat(all_data, ignore_index=True)
    df_final.to_csv(fname, sep=';', index=False)

    return df_final

In [5]:

resource_gen: Generator[TabularResource] =scraper()
df = gerar_df_appggs(resource_gen)

Abaixo checamos que funcionou corretamente pois os primeiros APPGGs entraram em exercício em junho de 2016 que é o valor mínimo para a data de referência no dataframe.

In [6]:
df['data_referencia'].min()

'2016-06-01'

Mas parece haver um problema de falta de padronização nas colunas.

In [7]:
df.columns

Index(['REGISTRO', 'VINCULO', 'NOME', 'CARGO_BASICO', 'REF_CARGO_BAS',
       'SEGMENTO', 'GRUPO', 'SUBGRUPO', 'ESCOL_CARGO_BASICO', 'CARGO_COMISSAO',
       'REF_CARGO_COM', 'ESCOL_CARGO_COMISSAO', 'DESIGNACAO', 'REF_DESIGNACAO',
       'ESCOL_DESIGNACAO', 'JORNADA', 'DATA_INICIO_EXERC', 'REL_JUR_ADM',
       'SECRET_SUBPREF', 'SIGLA', 'SETOR', 'DISTRITO', 'SUBPREFEITURA',
       'ORGAO_EXT', 'SEXO', 'ANO_NASCIMENTO', 'RACA_COR', 'PCD',
       'data_referencia', 'ano_referencia', 'mes_referencia', 'RACA',
       'DEFICIENTE', 'DATA_NASCIMENTO', 'Exceção', 'Nome completo',
       'Cargo Base', 'Cargo em Comissão', 'Remuneração do Mês',
       'Demais Elementos da Remuneração', 'Remuneração Bruta', 'Unidade',
       'Tp. Log', 'Logadrouro', 'Número', 'Complemento', 'Jornada',
       'Unnamed: 20', 'ESCOL_CARGO_BASE', 'SECRET_PREFREG'],
      dtype='str')

Como vemos abaixo um dataset veio com nomes de colunas diferentes (por exemplo, veio sem a coluna REGISTRO). Abaixo checamos que se trata do dataframe de janeiro de 2022 que parece estar fora do padrão (ele parece na verdade ser um recurso do dataset de remuneraçao dos servidores e nao dos servidores ativos - alguém deve ter se confundido quando fez o carregamento.)

In [8]:
df['REGISTRO'].isnull().sum()

np.int64(99)

In [9]:
df[df['REGISTRO'].isnull()]['data_referencia'].unique()

<StringArray>
['2022-01-01']
Length: 1, dtype: str

Abaixo checamos que essas colunas diferentes estão preenchidas apenas para o mes de referencia de janeiro de 2022.

In [10]:
for col in df.columns:
    if col.istitle() or col[0].isupper() and col[1].islower():
        print(col)
        meses_referencia_col_tem_dados = df[df[col].notnull()]['data_referencia'].unique()
        for mes in meses_referencia_col_tem_dados:
            print(mes)

Exceção
Nome completo
2022-01-01
Cargo Base
2022-01-01
Cargo em Comissão
2022-01-01
Remuneração do Mês
2022-01-01
Demais Elementos da Remuneração
2022-01-01
Remuneração Bruta
2022-01-01
Unidade
2022-01-01
Tp. Log
2022-01-01
Logadrouro
2022-01-01
Número
2022-01-01
Complemento
2022-01-01
Jornada
2022-01-01
Unnamed: 20


A primeira coisa que precisamos fazer para arrumar essa situação é recuperar a informação do registro (RF) que está ausente em 2022. Para isso vamos usa a informação dos outros anos. Vou ordenara o dataframe para ficar com os registros mais recentes no topo assim sabemos qual o nome mais atualizado da pessoa.

In [11]:
df = df.sort_values(by='data_referencia', ascending=False).reset_index(drop=True)

In [12]:
registros_nomes = df['REGISTRO'].apply(lambda x: str(int(x)) if not pd.isnull(x) else '').str.strip() + ':' + df['NOME'].fillna('').str.strip()

In [13]:
registros_nomes = registros_nomes.unique()

In [14]:
#checando pessoas que por acaso tiveram mudanças nos nomes para padronizar
# como o dataframe está com ordenado pelo ano mais recente o primeiro nome a aparecer para um dado RF é o nome mais atual
dici_rfs = {}
dici_nomes = {}
for par in registros_nomes:
    rf, nome = par.split(':')
    if rf not in dici_rfs:
        dici_rfs[rf] = nome
    if dici_rfs[rf] != nome:
        print(f'Inconsistência encontrada para RF {rf}: {dici_rfs[rf]} vs {nome}')
    
    if nome not in dici_nomes:
        dici_nomes[nome] = rf

Inconsistência encontrada para RF 8892121: ANDRE DA SOLLER vs ANDRE DA SOLER
Inconsistência encontrada para RF 8414611: MARIA CAMILA FLORENCIO DOTTO vs MARIA CAMILA FLORENCIO DA SILVA
Inconsistência encontrada para RF 8405638: DEBORA GAMBETTA PEREIRA PAIM vs DEBORA GAMBETTA PAIM
Inconsistência encontrada para RF 8359466: RITA DE CASSIA DA CRUZ SILVA MINVIELLE vs RITA DE CASSIA DA CRUZ SILVA
Inconsistência encontrada para RF 8359440: MIRIA GOMES DO NASCIMENTO vs MIRIÃ GOMES DO NASCIMENTO
Inconsistência encontrada para RF 8235112: TERRA JOHARI POSSA TERRA vs JOAO PAULO POSSA TERRA
Inconsistência encontrada para RF 8359172: GUSTAVO GUIMARAES DE CAMPOS RABELLO vs GUSTAVO GUIMARÃES DE CAMPOS RABELLO
Inconsistência encontrada para RF 8359008: MARCO AURELIO LESSA VILLELA vs MARCO AURÉLIO LESSA VILLELA
Inconsistência encontrada para RF 8358982: FERNANDO HIDEKI ISHIDA OSHIMA vs FERNANDO HIDEKI  ISHIDA OSHIMA


Vamos trocar os nomes para deixar o nome padronizado. O nome mais recente é o mais atual: temos alguns casos de acentuação que nos registros masi recente foi retirada mas também temos casos de nomes de solteiro/casados então é bom pegar o nome mais recente.

Mas antes de padronizar o nome preciso padronizar a coluna de registro. Já vou salvando com os nomes de colunas que vou usar depois.

In [15]:
df['rf'] = df['REGISTRO'].apply(lambda x: str(int(x)) if not pd.isnull(x) else None)

Consigo pegar os rfs das pessoas para janeiro de 2022 a partir do nome delas

In [16]:
df[df['rf'].isnull()]['Nome completo']

6496            BRENDA MACHADO FONSECA
6497                  BRUNO MARTINELLI
6498             GABRIELA SANTOS NEVES
6499    JARBAS ANTONIO DE BIAGI JUNIOR
6500          LEONARDO SPICACCI CAMPOS
                     ...              
6590          THIAGO FERREIRA DE SOUZA
6591            RAMON SANTORO LEONARDI
6592                 THOR SAAD RIBEIRO
6593            VANESSA SILVA CARVALHO
6594           VINICIUS PEDRON MACARIO
Name: Nome completo, Length: 99, dtype: str

In [17]:
def padronizar_rf(row):

    if pd.isnull(row['rf']):
        nome = row['Nome completo']
        if nome in dici_nomes:
            return dici_nomes[nome]
        else:
            if nome =='SERVIDOR PUBLICO MUNICIPAL':
                return None
            raise ValueError(f'Pessoa não encontrada: {nome}')
    return row['rf']

In [18]:
df['rf'] = df.apply(padronizar_rf, axis=1)

In [19]:
#ficamos com 3 casos de registros nulos por conta do nome genérico "SERVIDOR PUBLICO MUNICIPAL" que não conseguimos identificar
#esse nome é usado pela secretaria de segurança urbana para anonimizar a base
df['rf'].isnull().sum()

np.int64(3)

In [20]:
df[df['rf'].isnull()]['Nome completo']

6526    SERVIDOR PUBLICO MUNICIPAL
6527    SERVIDOR PUBLICO MUNICIPAL
6545    SERVIDOR PUBLICO MUNICIPAL
Name: Nome completo, dtype: str

In [21]:
#vamos ter que dropar esses registros porque não tem como identificar a pessoa

df = df[df['rf'].notnull()].reset_index(drop=True)

In [22]:
#Agora podemos padronizar o nome das pessoas com o outro dicionario que criamos que tem o nome mais recente

In [23]:
df['nome'] = df['rf'].map(dici_rfs)

In [24]:
df['nome'].isnull().any()

np.False_

In [25]:
df['nome'].str.lower().str.contains('servidor').any()

np.False_

In [26]:
#checando se deu certo - só pode ter uma única pessoa chamada Terra
df[df['nome'].str.lower().str.contains('terra')]['nome'].unique()

<StringArray>
['TERRA JOHARI POSSA TERRA']
Length: 1, dtype: str

Agora precisamos preencher o cargo que a pessoa ocupava em janeiro de 2022. Nesse caso vamos pegar o cargo que ela ocupava no mes anterior (dezembro de 2021) e preencher.

In [27]:
df['REF_CARGO_BAS'].isnull().sum()

np.int64(96)

In [28]:
def padronizar_cargo(row, df:pd.DataFrame):

    cargo_base = row['REF_CARGO_BAS']
    if not pd.isnull(cargo_base):
        return cargo_base
    
    rf = row['rf']
    dt_dez_2021 = '2021-12-01'
    cargo_mes_anterior = df[(df['rf']==rf)&(df['data_referencia']==dt_dez_2021)]['REF_CARGO_BAS'].values[0]
    return cargo_mes_anterior
    

In [29]:
df['cargo_base'] = df.apply(padronizar_cargo, axis=1, df=df)

In [30]:
df['cargo_base'].unique()

<StringArray>
['APPGG2', 'APPGG1', 'APPGG3', 'APPGG6', 'APPGG5', 'APPGG4']
Length: 6, dtype: str

Basicamente precisamos apenas desses dados - nome da pessoa, RF, e cargo base que ela ocupava em um determinado momento (e é claro o dado de qual momento era esse). Com esse dados conseguimos aplicar a tabela de vencimentos que estava valendo na época e calcular o vencimento no momento. Depois disso calcular qual deveria ser esse vencimento considerando a tabela original ajustada pelo IPC acumulado até aquela época. Aí temos para cada momento qual a perda. No final podemos atualizar essa perda também para valor corrente usando o ICP (o IPC é o índice oficial da Prefeitura)

In [31]:
colunas_interesse = ['rf', 'nome', 'cargo_base', 'data_referencia', 'ano_referencia', 'mes_referencia']

In [32]:
df = df[colunas_interesse].copy(deep=True)

In [33]:
df.head()

,rf,nome,cargo_base,data_referencia,ano_referencia,mes_referencia
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,2026-02-01,2026,2
1,8915369,OG OLIVEIRA PINTO,APPGG2,2026-02-01,2026,2
2,8915164,FLAVIO FERNANDES,APPGG2,2026-02-01,2026,2
3,8915181,JOSE CARLOS CALLEGARI,APPGG2,2026-02-01,2026,2
4,8915211,LUCAS DE CASTRO SENA,APPGG2,2026-02-01,2026,2


In [34]:
for col in df.columns:
    assert df[col].notnull().all(), f'Coluna {col} tem valores nulos'

In [35]:
df['nivel'] = df['cargo_base'].str[-1].astype(int)

In [36]:
#está certo os APPGGs mais antigos estão no nivel 6
df['nivel'].unique()

array([2, 1, 3, 6, 5, 4])

Abaixo vou registrar os dicionarios que representam as tabelas

In [37]:
original = {
    1 : 9000,
    2 : 10080,
    3 : 10684.80,
    4 : 11325.89,
    5 : 12005.44,
    6 : 12725.77,
    7 : 13998.34,
    8 : 14698.26,
    9 : 15433.17,
    10 : 16204.83,
    11 : 17015.08,
    12 : 18716.58,
    13 : 19558.83,
    14 : 20438.98,
    15 : 21358.73
}

In [38]:
maio_22 = {
    1 : 12000,
    2 : 13200,
    3 : 13530,
    4 : 13868.25,
    5 : 14214.96,
    6 : 14570.33,
    7 : 16318.77,
    8 : 16726.74,
    9 : 171444.91,
    10 : 17573.53,
    11 : 18012.87,
    12 : 19869.32,
    13 : 20366.05,
    14 : 20875.20,
    15 : 21397.08
}

In [39]:
maio_23 = {
    1 : 12601.26,
2 : 13861.38,
3 : 14207.91,
4 : 14563.11,
5 : 14927.19,
6 : 15300.36,
7 : 17136.42,
8 : 17564.83,
9 : 18003.95,
10 : 18454.04,
11 : 18915.40,
12 : 20864.86,
13 : 21386.48,
14 : 21921.14,
15 : 22469.17
}

In [40]:
maio_24= {
    1 : 12873.44,
2 : 14160.78,
3 : 14514.80,
4 : 14877.67,
5 : 15249.61,
6 : 15630.84,
7 : 17506.56,
8 : 17944.23,
9 : 18392.83,
10 : 18852.64,
11 : 19323.97,
12 : 21315.54,
13 : 21848.42,
14 : 22394.63,
15 : 22954.50
}

In [41]:
maio_25 = {
1  : 13208.14,
2 :  14528.96,
3 :  14892.18,
4 :  15264.48,
5 :  15646.09,
6 : 16037.24,
7 :  17961.73,
8 :  18410.77,
9  : 18871.04,
10 :  19342.80,
11 : 19826.39,
12 : 21869.74,
13 : 22416.47,
14:  22976.89,
15 : 23551.31
}


In [42]:
reajuste_26 = 0.0255
maio_26 = {key : round(value * (1+reajuste_26), 2) for key, value in maio_25.items()}
maio_26

{1: 13544.95,
 2: 14899.45,
 3: 15271.93,
 4: 15653.72,
 5: 16045.07,
 6: 16446.19,
 7: 18419.75,
 8: 18880.24,
 9: 19352.25,
 10: 19836.04,
 11: 20331.96,
 12: 22427.42,
 13: 22988.09,
 14: 23562.8,
 15: 24151.87}

In [43]:
tabelas_ano = {
    2016  : original,
    2022 : maio_22,
    2023 : maio_23,
    2024 : maio_24,
    2025 : maio_25,
    2026 : maio_26
    }

In [44]:
def pegar_tabela_ano_mes(ano:int, mes:int)->float:


    if ano < 2016:
        raise ValueError('Não deveria ter ano antes de 2016')
    
    if mes < 5:
        #tem que voltar um ano porque aí a tabela de maio do ano anterior ainda é válida
        ano -= 1

    if ano < 2022:
        #aí retorna a original
        return tabelas_ano[2016]
    #se nao retorna a tabela daquele ano (lembrando que já ajustamos a questão do mês de maio)
    return tabelas_ano[ano]
    

    
def pegar_salario_tabela(nivel:int, ano:int, mes:int)->float:

    if nivel < 0 or nivel > 15:
        raise ValueError('Nivel deve ser entre 1 e 15')

    tabela = pegar_tabela_ano_mes(ano, mes)

    return tabela[nivel]

In [45]:
df['vencimento_nominal'] = df.apply(lambda row: pegar_salario_tabela(row['nivel'], row['ano_referencia'], row['mes_referencia']), axis=1)

In [46]:
df['vencimento_nominal'].sample()

1163    16037.24
Name: vencimento_nominal, dtype: float64

In [47]:
df[df['nome'].str.lower().str.contains('pougy')].sample(5)

,rf,nome,cargo_base,data_referencia,ano_referencia,mes_referencia,nivel,vencimento_nominal
4507,8359164,HENRIQUE POUGY,APPGG4,2023-04-01,2023,4,4,13868.25
5771,8359164,HENRIQUE POUGY,APPGG3,2022-07-01,2022,7,3,13530.00
7857,8359164,HENRIQUE POUGY,APPGG2,2020-08-01,2020,8,2,10080.00
4915,8359164,HENRIQUE POUGY,APPGG4,2023-01-01,2023,1,4,13868.25
3116,8359164,HENRIQUE POUGY,APPGG4,2024-02-01,2024,2,4,14563.11


Agora podemos pegar os valores do salario original atualizado para a inflacao naquela data de referenica.

Para isso vamos usar uma classe que puxa os dados da api do Bacen (a classe usa memoizacao para não ficar buscando toda hora e otimizar)

In [48]:
from core.reajuste import CalculadoraFatorInflacao

In [49]:
reajuste_ipc = CalculadoraFatorInflacao(indice='ipc-fipe')

In [50]:
data_inicial_carreira = '01/06/2016'

In [51]:
def repadronizar_data_referencia(data:str)->str:

    ano, mes, dia = data.split('-')
    return f'{dia}/{mes}/{ano}'

In [52]:
#precisa repadronizar a data de referencia para funcionar com o calculador de inflacao
df['data_referencia'] = df['data_referencia'].apply(repadronizar_data_referencia)
df['data_referencia'].sample(5)

1       01/02/2026
7547    01/01/2021
9253    01/01/2019
5526    01/08/2022
3012    01/02/2024
Name: data_referencia, dtype: str

In [53]:
def vencimentos_originais_atualizados(nivel:int, data_referencia:str)->float:

    fator = reajuste_ipc(data_inicial=data_inicial_carreira, data_final=data_referencia)
    tabela_original = tabelas_ano[2016]
    vencimento_original = tabela_original[nivel]
    vencimento_atualizado = round(vencimento_original * fator, 2)
    return vencimento_atualizado

In [54]:
try:
    df['vencimento_original_atualizado'] = df.apply(lambda row: vencimentos_originais_atualizados(row['nivel'], row['data_referencia']), axis=1)
except Exception as e:
    print(f'Erro ao calcular vencimento original atualizado: {e}')
    reajuste_ipc.save_indices()
    raise e

In [55]:
reajuste_ipc.save_indices()

In [56]:
df.sample(5)

,rf,nome,cargo_base,data_referencia,ano_referencia,mes_referencia,nivel,vencimento_nominal,vencimento_original_atualizado
5832,8915393,ANDREA GIANNELLA BANDEIRA,APPGG1,01/06/2022,2022,6,1,12000.00,12343.16
10672,8358826,LEONARDO SPICACCI CAMPOS,APPGG1,01/06/2017,2017,6,1,9000.00,9282.59
4523,8582629,LUCCAS BERNACCHIO GISSONI,APPGG1,01/03/2023,2023,3,1,12000.00,12757.60
2532,8111618,VINICIUS PEDRON MACARIO,APPGG5,01/06/2024,2024,6,5,15249.61,17626.25
11068,8280266,MONICA DE AZEVEDO COSTA NOGARA,APPGG1,01/12/2016,2016,12,1,9000.00,9191.37


In [57]:
df['perda_inflacionaria_nominal'] = df['vencimento_original_atualizado'] - df['vencimento_nominal']

Vamos checar se em algum momento para algum appgg houve "ganho" real acima da inflacao. Como vemos abaixo nao houve (note que o sinal é invertido porque é uma "perda", por isso o ganho seria a perda negativa)

In [58]:
df[df['perda_inflacionaria_nominal']<0]

,rf,nome,cargo_base,data_referencia,ano_referencia,mes_referencia,nivel,vencimento_nominal,vencimento_original_atualizado,perda_inflacionaria_nominal


Agora falta só atualizar a perda inflacionaria nominal para valores correntes

In [59]:
from datetime import datetime

In [60]:
hoje_formatado = datetime.now().strftime("%d/%m/%Y")

In [61]:
def perda_inflacionaria_atualizada(perda_inflacionaria_nominal:float, data_referencia:str)->float:

    fator = reajuste_ipc(data_inicial=data_referencia, data_final=hoje_formatado)
    perda_atualizada = round(perda_inflacionaria_nominal * fator, 2)
    return perda_atualizada

In [62]:
df['perda_inflacionaria_atualizada'] = df.apply(lambda row: perda_inflacionaria_atualizada(row['perda_inflacionaria_nominal'], 
                                                                                           row['data_referencia']), axis=1)

Dados obtidos com sucesso na URL: https://api.bcb.gov.br/dados/serie/bcdata.sgs.193/dados?formato=json&dataInicial=01%2F02%2F2026&dataFinal=07%2F04%2F2026
Dados obtidos com sucesso na URL: https://api.bcb.gov.br/dados/serie/bcdata.sgs.193/dados?formato=json&dataInicial=01%2F01%2F2026&dataFinal=07%2F04%2F2026
Dados obtidos com sucesso na URL: https://api.bcb.gov.br/dados/serie/bcdata.sgs.193/dados?formato=json&dataInicial=01%2F12%2F2025&dataFinal=07%2F04%2F2026
Dados obtidos com sucesso na URL: https://api.bcb.gov.br/dados/serie/bcdata.sgs.193/dados?formato=json&dataInicial=01%2F11%2F2025&dataFinal=07%2F04%2F2026
Dados obtidos com sucesso na URL: https://api.bcb.gov.br/dados/serie/bcdata.sgs.193/dados?formato=json&dataInicial=01%2F10%2F2025&dataFinal=07%2F04%2F2026
Dados obtidos com sucesso na URL: https://api.bcb.gov.br/dados/serie/bcdata.sgs.193/dados?formato=json&dataInicial=01%2F09%2F2025&dataFinal=07%2F04%2F2026
Dados obtidos com sucesso na URL: https://api.bcb.gov.br/dados/serie/b

In [63]:
reajuste_ipc.save_indices()

In [64]:
df['perda_inflacionaria_atualizada'].sample(5)

10647     433.93
10752     426.60
8252     1624.74
7153     2985.18
5839      397.20
Name: perda_inflacionaria_atualizada, dtype: float64

In [65]:
df.sample(5)

,rf,nome,cargo_base,data_referencia,ano_referencia,mes_referencia,nivel,vencimento_nominal,vencimento_original_atualizado,perda_inflacionaria_nominal,perda_inflacionaria_atualizada
6232,8894302,PAULA HELOISA DA SILVA RIBEIRO,APPGG1,01/03/2022,2022,3,1,9000.00,12061.81,3061.81,3662.76
9473,8359512,ESTEVAO NICOLAU RABBI DOS SANTOS,APPGG1,01/10/2018,2018,10,1,9000.00,9659.22,659.22,976.98
8931,8359237,MARILIA ROMAO CAPINZAIKI,APPGG1,01/05/2019,2019,5,1,9000.00,9867.66,867.66,1252.47
1688,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,01/01/2025,2025,1,2,14160.78,15243.77,1082.99,1136.35
3679,8359156,LARISSA DIANA MICHELAM,APPGG4,01/10/2023,2023,10,4,14563.11,16191.24,1628.13,1808.25


In [66]:
#arredondando as colunas de dinheiro para deixar com 2 casas decimais

colunas_dinheiro = ['vencimento_nominal', 'vencimento_original_atualizado', 'perda_inflacionaria_nominal', 'perda_inflacionaria_atualizada']

for col in colunas_dinheiro:
    df[col] = df[col].apply(lambda x: round(x, 2))

In [67]:
df.to_csv('microdados_perda_inflacionaria.csv', sep=';', decimal =',' ,index=False)

In [68]:
total_por_appgg = df.groupby('nome')[['perda_inflacionaria_nominal', 'perda_inflacionaria_atualizada']].sum().reset_index().sort_values(by='perda_inflacionaria_atualizada', ascending=False)

In [69]:
total_por_appgg

,nome,perda_inflacionaria_nominal,perda_inflacionaria_atualizada
50,DIEGO XAVIER LEITE,182451.40,218683.89
202,THIAGO FERREIRA DE SOUZA,181836.13,218005.51
124,LUCAS AMBROZIO LOPES DA SILVA,181836.13,218005.51
101,JOAO PAULO DE BRITO GRECO,181723.49,217854.01
203,THIAGO LUIZ ROSASCO ERMEL,181723.49,217854.01
...,...,...,...
161,MURILO PAIVA D AGOSTA,955.17,963.21
191,SILVIO LUIZ VENTAVELE DA SILVA,955.17,963.21
196,TANIELI DE MORAES GUIMARAES SILVA,955.17,963.21
214,VITORIA CALDAS DO NASCIMENTO,955.17,963.21


In [72]:
for col in colunas_dinheiro:
    if col in total_por_appgg.columns:
        total_por_appgg[col] = total_por_appgg[col].apply(lambda x: round(x, 2))

In [73]:
total_por_appgg.to_csv('perda_total_por_appgg.csv', sep=';', decimal=',', index=False)

In [74]:
total_por_appgg[total_por_appgg['nome'].str.contains('POUGY')]

,nome,perda_inflacionaria_nominal,perda_inflacionaria_atualizada
88,HENRIQUE POUGY,174170.68,209459.28
